In [ ]:
# Imports

# Import Python packages
import os

import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
from pprint import pprint
from copy import deepcopy

# Import the Simulation packages
from spinfinity import *
from scipy.io import loadmat

# Procedure
- Load the Experiment
    - Potential Data Processing
- Simulation by Hand
    - Build the Spin System
    - Find Initial Guess
    - set Boundaries
- Build the Fitfunction
    - Just change the relevant parameter 
    - Just compute the relevant stuff
    ! Make sure everything else is set beforehand 
- Fit & Plot Result

# Experiment

In [ ]:
# get notebook directory and navigate to 'testdata' (create if missing)
nb_dir = os.getcwd()
testdata_dir = os.path.join(nb_dir, "testdata")

# Load the data files
filename1= "AVG_Spec20230402_055608_Spec20230402_070414.mat" #Complex Center 
filename2= "AVG_Spec20230401_031946_Spec20230401_044554.mat" #Complex Fe
filepath1=os.path.join(testdata_dir,filename1)
filepath2=os.path.join(testdata_dir,filename2)
data1 = loadmat(filepath1)
data2 = loadmat(filepath2)

# Extract x and y data
x1=data1['Bias'][0]
y1=data1['lockinX'][0]/np.max(data1['lockinX'][0])
x2=data2['Bias'][0]
y2=data2['lockinX'][0]/np.max(data2['lockinX'][0])

# Plot the data
plt.figure(figsize=(8,5))
plt.plot(x1, y1, '-', lw=1, color='red', label='Complex Center')
plt.plot(x2, y2, '-', lw=1, color='blue', label='Complex Fe')
plt.legend()
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (A)')
plt.grid(True)
plt.tight_layout()
plt.show()


# Simulation

In [ ]:
# Define the Spin System
spinsys=SpinSys([1,0.5])

# Basic Parameters
spinsys.B=[0,0,0.1]
spinsys.BTip=[1e-9,0,0]
spinsys.tip='f'
spinsys.T=0.2

# Coupling Parameters
spinsys.Jvector=np.array([[14,14,14]])
spinsys.Dvector=np.array([1.9,0])
spinsys.AtomPositions=np.array([[0,0,0],[2,1,0]])
spinsys.DipoleBool=[False]

# Tunnel Parameters
spinsys.V_DC=100
spinsys.TipPolarization = [0,0,0]
spinsys.U=[0, 0]
spinsys.G_st=1e-10
tsratio=1
spinsys.G_ss=[spinsys.G_ss*tsratio, spinsys.G_ss*tsratio]
spinsys.G_tt=1

# ESR Parameters
spinsys.ReadoutSpin=1
spinsys.RabiRate=1
spinsys.T2=200


dIdV.plot_IETS(spinsys, V_array=np.linspace(-30,30,1000), norm=True, order='2nd', plotAllSpins=True);

# Fit

In [ ]:
# Define the fitting function 
def fit_fun(x_vals, U, T, J, D): # x and Open variables

    # Modify the spin system according to the Variables
    spinsys.U[spinsys.ReadoutSpin]=U
    spinsys.T=T
    spinsys.Jvector=np.array([[J,J,J]])
    spinsys.Dvector=np.array([D,0])

    
    
    # Calculate the Energies new
    hamil.calc_EigEnergies(spinsys)
    hamil.calc_EigStates(spinsys)

    # Calculate the dIdV spectrum according 
    N=len(x_vals)
    V, I=dIdV.calc_IETS_2nd(spinsys, V_array=np.linspace(np.min(x_vals),np.max(x_vals),N), norm=True);
    return I

from scipy.optimize import curve_fit


# Initial guess:
initial_guess = [0,1,20,5]

# Bounds:
bounds_custom = [(-1, 1),  # U
                 (0.05,3),    # T
                 (10, 21),    # J
                 (1,10),    # D
                 ]

# Choose which data to fit
spinsys.ReadoutSpin=1 # For the Fe
x=x1
y=y1

# Perform the fit
popt, pcov = curve_fit(fit_fun, x, y, p0=initial_guess, bounds=([b[0] for b in bounds_custom], [b[1] for b in bounds_custom]))

# Print the fit results
pprint(popt)
perr = np.sqrt(np.diag(pcov))
pprint(perr)

## Plot the Fit

In [ ]:
# Calculate fitted curve
x_fit = np.linspace(np.min(x), np.max(x), 1000)
y_fit=fit_fun(x_fit, *popt)

# Plot the results
plt.figure(figsize=(8,5))
plt.plot(x, y, 'k.', markersize=4)
plt.plot(x_fit, y_fit, 'r-', lw=1)
label = (
    "fit:\n"
    f"U={popt[0]:.3g} ± {perr[0]:.3g}\n"
    f"T={popt[1]:.3g} ± {perr[1]:.3g}\n"
    f"J={popt[2]:.3g} ± {perr[2]:.3g}\n"
    f"D={popt[3]:.3g} ± {perr[3]:.3g}"
)
plt.legend(['data', label], loc='best')
plt.xlabel('Voltage (mV)')
plt.ylabel('Current (A)')
plt.grid(True)
plt.tight_layout()
plt.show()